# BERT / GELECTRA Implementation

## 1. Environment Setup

### 1.1. Imports

In [1]:
## Importing DS modules
import pandas as pd

## Importing Torch modules
import torch

## Importing HF modules
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

## Importing LangChain modules
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma

## Importing other modules
import copy
import os

### 1.2 Global Variables

In [2]:
## Path for documents to be retrieved
DOCUMENT_PATH = "/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/02 Group project/data/raw_data"

## Path for finetuned_model
FINETUNED_MODEL_PATH = '/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/02 Group project/bert/bert-finetuned'

## Path for test data 
TEST_PATH = "/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/02 Group project/evaluation/final_testset.csv"

## Path to save result on test
TEST_SAVE_PATH = "/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/02 Group project/evaluation"

In [3]:
## Initialization of Torch device to use CUDA for GPU acceleration for faster computation
DEVICE = torch.device("mps")

## 2.0 Retriever Setup

In [6]:
## Loading documents to be retrieved
loader = PyPDFDirectoryLoader(DOCUMENT_PATH)
documents = loader.load()

In [8]:
from haystack.document_stores import FAISSDocumentStore
document_store = FAISSDocumentStore(faiss_index_factory_str="Flat")

In [10]:
# Import the necessary modules and functions from Haystack
from haystack.utils import convert_files_to_docs
from haystack.nodes import PreProcessor, TextConverter, PDFToTextConverter, DocxToTextConverter

# Converters are used to convert documents into text
# Here we are using a PDFToTextConverter to convert PDF documents to text
converter = PDFToTextConverter(
    remove_numeric_tables=True,  # If True, removes tables that consist of only numbers
    #valid_languages=["de"]      # Only documents in German ("deu") should be converted
)

# Use the converter to convert a specific PDF file into text
doc_pdf = converter.convert(file_path="/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/02 Group project/data/raw_data/mbs/all.pdf", meta=None)[0]  # Metadata for the document. If None, no metadata will be added
# The convert method returns a list of documents, we're only interested in the first one

# PreProcessor is used to perform preprocessing on the text
preprocessor_sliding_window = PreProcessor(
    split_overlap=100,  # Number of words that the resulting chunks will overlap
    split_length=400,   # Number of words in one chunk
    split_respect_sentence_boundary=True,  # If True, splits would be made at sentence boundaries
    clean_empty_lines=True,  # If True, normalizes 3 or more consecutive empty lines to be just two empty lines
    clean_whitespace=True,  # If True, removes any whitespace at the beginning or end of each line in the text
    clean_header_footer=False,  # If True, removes any long header or footer texts that are repeated on each page
    #split_by="word",  # Document is split on word boundaries
    language="de",  # Language of the document
)

# Use the preprocessor to process the converted document
docs_sliding_window = preprocessor_sliding_window.process([doc_pdf])

# Write the processed documents to a Document Store
# Please note that the document_store object must be already initialized in the scope
document_store.write_documents(docs_sliding_window)

pdftotext version 4.04 [www.xpdfreader.com]
Copyright 1996-2022 Glyph & Cog, LLC


Preprocessing:   0%|          | 0/1 [00:00<?, ?docs/s]

We found one or more sentences whose word count is higher than the split length.


Writing Documents:   0%|          | 0/876 [00:00<?, ?it/s]

In [12]:
from haystack.nodes import EmbeddingRetriever

embedding_retriever = EmbeddingRetriever(
    document_store=document_store,
    embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
)

# Alternatives: https://www.sbert.net/docs/pretrained_models.html#

# distiluse-base-multilingual-cased-v1

#all-mpnet-base-v2
#paraphrase-multilingual-mpnet-base-v2
#all-distilroberta-v1

# Update embeddings
document_store.update_embeddings(embedding_retriever)

Updating Embedding:   0%|          | 0/876 [00:00<?, ? docs/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

## 3.0 Pipeline Setup

In [21]:
## Instantiating finetuned model & tokenizer
model = AutoModelForQuestionAnswering.from_pretrained('/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/02 Group project/bert/bert-finetuned')
#model_notft = AutoModelForQuestionAnswering.from_pretrained("deepset/gelectra-large-germanquad")
tokenizer = AutoTokenizer.from_pretrained("deepset/gelectra-large-germanquad")

model.to(DEVICE)
#model_notft.to(DEVICE)

ElectraForQuestionAnswering(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-23): 24 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerN

In [22]:
## Setting up pipeline for question-answering
qa_pipeline = pipeline('question-answering', model=model, tokenizer=tokenizer, device="mps")

In [23]:
## Setting example query
query = " Wie lange darf man bei der Arbeit Pausen machen?"

In [24]:
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(model, embedding_retriever)

# You can configure how many candidates the reader and retriever shall return
# The higher top_k for retriever, the better (but also the slower) your answers.
prediction = pipe.run(
    query=query, params={"Retriever": {"top_k": 3}, "Reader": {"top_k": 1}}
)

from haystack.utils import print_answers
print_answers(prediction, details="medium")

AttributeError: 'ElectraForQuestionAnswering' object has no attribute '_component_config'

In [ ]:
## Function to retrieve context and answer from model
def query_model(pipeline, vectorstore, query):
    context = vectorstore.similarity_search(query)[0].page_content
    answer = qa_pipeline({'context': context, 'question': query})
    full_answer = answer["answer"]
    return context, full_answer

## Prompting our model based on example query
query_model(qa_pipeline, vs, query)

In [15]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

from transformers import pipeline
from langchain.llms import HuggingFacePipeline
import torch

pipe = pipeline(
    "text-generation",
    model=model, 
    tokenizer=tokenizer, 
)

local_llm = HuggingFacePipeline(pipeline=pipe)

qa = ConversationalRetrievalChain.from_llm(local_llm, embedding_retriever, memory=memory)

The model 'ElectraForQuestionAnswering' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'GPT2LMHeadModel', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegatronBertForCausalLM', 'MvpForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusForCausalLM', 'PLBartForCausalLM', 'ProphetNetForCausalLM', 'QDQBertLMHeadModel', 'ReformerModelWithLMHead', 'RemBertForCausalLM', 'RobertaForCausalLM', 'RoCBertForCausalLM', 'RoFormerForCausalLM', 'Speech2Text2ForCausalLM', 'TransfoXLLMHeadModel', 'TrOCRForCausalLM', 'XGLMForCausalLM', 'XLMWithLMHeadModel', 'XLMProphetNetForCau

NameError: name 'ConversationalRetrievalChain' is not defined

In [152]:
## Function to retrieve context and answer from model
def query_model(pipeline, vectorstore, query):
    context = vectorstore.similarity_search(query)[0].page_content
    answer = qa_pipeline({'context': context, 'question': query})
    full_answer = answer["answer"]
    return context, full_answer

In [164]:
## Prompting our model based on example query
query_model(qa_pipeline, vs, query)

('ArG Art. 15Wegleitung zum Arbeitsgesetz\nIII. Arbeits- und Ruhezeit\n2. Ruhezeit\nArt. 15 Pausen\nAllgemeines\nDer Zweck der Pausen, die Erholung und die Ver -\npflegung, ist nur erfüllt, wenn sie etwa in der Mit -\nte der Arbeitszeit gewährt werden. «Pausen» am \nAnfang oder am Ende der Arbeitszeit sind keine \nechten Pausen und gelten nicht als gewährt (vgl. \nKommentar Art. 18 ArGV 1 ). Die aufgeführ -\nten Pausen bezeichnen Mindestwerte; eine länge -\nre Dauer der Pause kann jederzeit vereinbart wer -\nden.\nAbsatz 1\nBuchstabe a:\nBei einer Arbeitszeit von bis zu 5½ Stunden ist der \nArbeitgeber nicht verpflichtet, dem Arbeitnehmer \noder der Arbeitnehmerin eine Pause zu gewäh -\nren. Bei einer Arbeitszeit von über 5½ bis zu 7 \nStunden muss eine Pause von mindestens einer \nViertelstunde gewährt werden. Je nach der zwi -\nschen Arbeitsbeginn und Arbeitsende liegenden \nZeitspanne (Präsenzzeit) können sich Mindestpau -\nsen von anderer (kürzerer) Dauer als einer Viertel -\nstund

## 4.0 Model Evaluation (on test data)

In [155]:
## Retrieving test data
test = pd.read_csv(TEST_PATH)

In [157]:
## Name of model currently to be evaluated
model_name = "gelectra-notfinetuned-different-embedding"

In [158]:
## Collecting context & answers based on test data

contexts_model = []
answers_model = []

for row in test.itertuples():
    
    ## Retrieving question from test
    question = str(row.Frage)
    
    ## Generating context and answer based on pipeline
    context, answer = query_model(qa_pipeline_notft, vs, question)
    
    ## Appending context & answer to the list
    contexts_model.append(context)
    answers_model.append(answer)

In [159]:
## Adding generated contexts & answer to list 

evaluation = copy.deepcopy(test)
evaluation["ModelKontext"] = contexts_model
evaluation["ModelAntwort"] = answers_model

In [160]:
## Saving model to 
evaluation.to_csv(os.path.join(TEST_SAVE_PATH, f'{model_name}.csv'), index=False)